## HomeGuardian AI — remote certification (CPU session)

Builds and tests an **exact commit SHA** off the workstation. No credentials, no private media, no `.env`.
Set `HG_SHA` below (or pass via the local driver). Internet must be enabled for FetchContent deps (spdlog, nlohmann/json).

In [ ]:
#@title Parameters  { display-mode: "form" }
HG_SHA      = "REPLACE_WITH_SHA"          # exact 40-hex commit to certify
HG_REPO     = "https://github.com/methakon/homeguardian-ai.git"
HG_JOBS     = 0        # 0 = auto from available RAM (never nproc=40 blindly)
CM_JOBS_MIN = 3        # cmake must be >= 3.28.3 (project requirement)
RUN_TESTS   = True
RUN_CHECKERS= True

In [ ]:
import os, sys, json, subprocess, shutil, hashlib, time, textwrap, pathlib
def sh(cmd, cwd=None, check=False, env=None):
    t = time.time()
    p = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, capture_output=True, text=True, env=env)
    label = cmd if isinstance(cmd, str) else " ".join(cmd)
    print("$ " + label)
    print("  rc=" + str(p.returncode) + " (" + str(round(time.time() - t, 1)) + "s)")
    if p.stdout.strip():
        print(p.stdout[-4000:])
    if p.returncode and p.stderr.strip():
        print("STDERR: " + p.stderr[-2000:])
    if check and p.returncode:
        raise SystemExit("FAILED: " + label)
    return p

# ---- environment probe (this is also the smoke-test payload) ----
def mem_mb():
    for line in open("/proc/meminfo"):
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) // 1024
    return 0
import multiprocessing as mp
nproc = mp.cpu_count()
avail = mem_mb()
jobs = HG_JOBS or max(1, min(4, avail // 700))   # ~700MB per C++ TU: the rule learnt locally
probe = {"probe": "env", "nproc": nproc, "mem_available_mb": avail, "build_jobs": jobs,
         "python": sys.version.split()[0], "has_gpu": os.path.exists("/proc/driver/nvidia"),
         "internet": subprocess.run("getent hosts github.com >/dev/null 2>&1", shell=True).returncode == 0}
print(json.dumps(probe, indent=2))


In [ ]:
# ---- 1. exact-SHA checkout + repository identity ----
sh(f"rm -rf /kaggle/working/hg && git clone --quiet {HG_REPO} /kaggle/working/hg", check=True)
sh(f"git -C /kaggle/working/hg fetch --quiet --depth 1 origin {HG_SHA} && git -C /kaggle/working/hg checkout --quiet {HG_SHA}", check=True)
got = sh("git -C /kaggle/working/hg rev-parse HEAD").stdout.strip()
ident = sh("git -C /kaggle/working/hg remote get-url origin").stdout.strip()
assert got == HG_SHA, f"REVISION MISMATCH: wanted {HG_SHA}, got {got}"
print("REVISION IDENTITY OK:", got)

In [ ]:
# ---- 2. pinned toolchain (project needs cmake >= 3.28.3) ----
v = sh("cmake --version").stdout.split()[2] if shutil.which("cmake") else "0"
def tup(s): return tuple(int(x) for x in s.split('.')[:3] if x.isdigit())
if tup(v) < (3,28,3):
    sh("pip install --quiet 'cmake>=3.28.3,<4'")
    os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
need = tup(sh("cmake --version").stdout.split()[2])
if need < (3,28,3): sh("apt-get install -y -qq cmake", check=False)
print("cmake:", sh("cmake --version").stdout.split()[2], "| gcc:", sh("gcc --version").stdout.split()[3])

In [ ]:
# ---- 3. configure + build (bounded jobs, serialized: one build only) ----
repo="/kaggle/working/hg"; bdir=f"{repo}/build_remote"
t0=time.time()
sh(f"cmake -S {repo} -B {bdir} -DCMAKE_BUILD_TYPE=Release -DFETCHCONTENT_FULLY_DISCONNECTED=OFF", check=True)
sh(f"cmake --build {bdir} -j{jobs}", check=True)
build_s = round(time.time()-t0,1)
print(f"BUILD OK in {build_s}s with -j{jobs}")
sh(f"ls -la {bdir} | head -20")

In [ ]:
# ---- 4. Catch2 suite (bounded), capture real counts ----
tc={}
if RUN_TESTS:
    p=sh(f"HG_SOURCE_DIR={repo} {bdir}/homeguardian_tests", check=False)
    import re
    m=re.search(r"All tests passed \((\d+) assertions in (\d+) test cases\)", p.stdout) or \
      re.search(r"assertions: (\d+).*?test cases: (\d+)", p.stdout, re.S)
    tc={"exit_code":p.returncode,
        "assertions":int(m.group(1)) if m else None, "test_cases":int(m.group(2)) if m else None,
        "raw_tail":p.stdout[-400:]}
    if p.returncode: tc["note"]="non-zero: suite failed or bounded-run hit a limit"
    sh(f"ctest --test-dir {bdir} --output-on-failure", check=False)
print(json.dumps(tc, indent=2))

In [ ]:

# ---- 5. security / provenance checkers (offline, stdlib only) ----
checks=[]
if RUN_CHECKERS:
    for name,args in [
      ("package",  f"python3 {repo}/scripts/check_release_package.py {repo} --repo {repo} --package-script {repo}/scripts/package_release.sh --json"),
      ("manifest", f"python3 {repo}/scripts/check_release_manifest.py {repo} --json"),
      ("claims",   f"python3 {repo}/scripts/check_submission_claims.py {repo} --release-ref {HG_SHA} --doc-scope submission --json"),
    ]:
        p_=sh(args, check=False)
        checks.append({"check":name,"exit_code":p_.returncode,"findings_tail":p_.stdout[-1500:]})
        time.sleep(0.2)
print(json.dumps([{"check":c["check"],"exit_code":c["exit_code"]} for c in checks], indent=2))


In [ ]:
# ---- 6. structured PASS/FAIL report + checksums -> /kaggle/working ----
import hashlib, glob
def sha256f(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20), b''): h.update(b)
    return h.hexdigest()
report={"schema":"hg-remote-certification/1","sha":HG_SHA,"repo":ident,"build_jobs":jobs,
        "nproc":nproc,"mem_available_mb":avail,"build_seconds":build_s,
        "tests":tc,"checkers":checks,
        "verdict": "PASS" if (tc.get("exit_code")==0 and all(c["exit_code"] in (0,2) for c in checks)) else "REVIEW"}
open("/kaggle/working/certification_report.json","w").write(json.dumps(report, indent=2))
# ship the build log + test binary fingerprint for independent review
shutil.copy(f"{bdir}/CMakeCache.txt","/kaggle/working/CMakeCache.txt")
arts={os.path.basename(f):sha256f(f) for f in
      ["/kaggle/working/certification_report.json","/kaggle/working/CMakeCache.txt"]}
binp=f"{bdir}/homeguardian_tests"
if os.path.exists(binp): arts["homeguardian_tests"]=sha256f(binp)
open("/kaggle/working/checksums.json","w").write(json.dumps(arts, indent=2))
print(json.dumps({"report":report["verdict"],"checksums":arts}, indent=2))

In [ ]:
# ---- 7. summary a reviewer can read without opening the notebook ----
print(textwrap.dedent(f'''
================= HOMEguardian REMOTE CERTIFICATION =================
commit            : {HG_SHA}
repository        : {ident}
runner            : kaggle cpu session  nproc={nproc}  mem_avail={avail}MB
build             : -j{jobs} in {build_s}s
tests             : {tc.get('test_cases')} cases / {tc.get('assertions')} assertions (exit {tc.get('exit_code')})
checkers          : {[(c['check'], c['exit_code']) for c in checks]}
VERDICT           : {report['verdict']}
artifacts         : /kaggle/working/certification_report.json, checksums.json, CMakeCache.txt
=====================================================================
''').strip())